# Tutorial: RL-01 Gridworld\n\nAudience:\n- Learners meeting tabular reinforcement learning for the first time.\n\nPrerequisites:\n- Markov decision processes, Bellman equations, and basic Python with NumPy.\n\nLearning goals:\n- Build a tiny deterministic environment.\n- Compare value iteration with model-free Q-learning.\n- Inspect learned value tables and greedy policies.\n

## Outline\n\n1. Define a small gridworld MDP\n2. Solve it with value iteration\n3. Learn it again from experience with Q-learning\n4. Compare the resulting greedy policies\n

In [ ]:
from __future__ import annotations\n\nfrom dataclasses import dataclass\nimport random\n\nimport numpy as np\n\nSEED = 7\nrandom.seed(SEED)\nnp.random.seed(SEED)\nSEED\n

## Step 1 - Define the environment\n\nThis gridworld has one goal state, one trap state, and a small step penalty so the agent prefers shorter paths.\n

In [ ]:
@dataclass\nclass GridWorld:\n    rows: int = 4\n    cols: int = 4\n    start: tuple[int, int] = (0, 0)\n    goal: tuple[int, int] = (3, 3)\n    trap: tuple[int, int] = (3, 1)\n    step_reward: float = -0.04\n    goal_reward: float = 1.0\n    trap_reward: float = -1.0\n\n    def __post_init__(self):\n        self.actions = ['up', 'right', 'down', 'left']\n        self.action_to_delta = {\n            'up': (-1, 0),\n            'right': (0, 1),\n            'down': (1, 0),\n            'left': (0, -1),\n        }\n        self.states = [(r, c) for r in range(self.rows) for c in range(self.cols)]\n        self.terminal_states = {self.goal, self.trap}\n        self.reset()\n\n    def reset(self):\n        self.state = self.start\n        return self.state\n\n    def is_terminal(self, state):\n        return state in self.terminal_states\n\n    def transition(self, state, action):\n        if self.is_terminal(state):\n            return state, 0.0, True\n        dr, dc = self.action_to_delta[action]\n        nr = min(max(state[0] + dr, 0), self.rows - 1)\n        nc = min(max(state[1] + dc, 0), self.cols - 1)\n        next_state = (nr, nc)\n        if next_state == self.goal:\n            return next_state, self.goal_reward, True\n        if next_state == self.trap:\n            return next_state, self.trap_reward, True\n        return next_state, self.step_reward, False\n\n    def step(self, action):\n        next_state, reward, done = self.transition(self.state, action)\n        self.state = next_state\n        return next_state, reward, done\n\nenv = GridWorld()\nenv.states[:5], env.actions\n

## Step 2 - Solve the model exactly with value iteration\n\nBecause the environment dynamics are known, we can repeatedly apply the Bellman optimality operator.\n

In [ ]:
gamma = 0.95\nvalues = {state: 0.0 for state in env.states}\n\nfor _ in range(200):\n    new_values = {}\n    for state in env.states:\n        if env.is_terminal(state):\n            new_values[state] = 0.0\n            continue\n        action_returns = []\n        for action in env.actions:\n            next_state, reward, done = env.transition(state, action)\n            bootstrap = 0.0 if done else values[next_state]\n            action_returns.append(reward + gamma * bootstrap)\n        new_values[state] = max(action_returns)\n    if max(abs(new_values[s] - values[s]) for s in env.states) < 1e-8:\n        values = new_values\n        break\n    values = new_values\n\ndef greedy_action_from_values(state, values_dict):\n    if env.is_terminal(state):\n        return 'TERM'\n    scores = {}\n    for action in env.actions:\n        next_state, reward, done = env.transition(state, action)\n        bootstrap = 0.0 if done else values_dict[next_state]\n        scores[action] = reward + gamma * bootstrap\n    return max(scores, key=scores.get)\n\nvalue_grid = np.array([[values[(r, c)] for c in range(env.cols)] for r in range(env.rows)])\npolicy_grid = np.array([[greedy_action_from_values((r, c), values) for c in range(env.cols)] for r in range(env.rows)])\nnp.round(value_grid, 3), policy_grid\n

## Interpretation\n\nThe values increase as states get closer to the goal and drop near the trap. This is the Bellman optimality equation in action: each state inherits the best downstream consequence of its actions.\n

## Step 3 - Learn from experience with Q-learning\n\nNow we pretend the model is unknown and estimate action values from sampled transitions.\n

In [ ]:
alpha = 0.2\nepsilon = 0.15\nepisodes = 600\nq_table = {(state, action): 0.0 for state in env.states for action in env.actions}\nepisode_returns = []\n\nfor _ in range(episodes):\n    state = env.reset()\n    done = False\n    total_reward = 0.0\n    while not done:\n        if random.random() < epsilon:\n            action = random.choice(env.actions)\n        else:\n            action = max(env.actions, key=lambda a: q_table[(state, a)])\n        next_state, reward, done = env.step(action)\n        best_next = 0.0 if done else max(q_table[(next_state, a)] for a in env.actions)\n        td_target = reward + gamma * best_next\n        td_error = td_target - q_table[(state, action)]\n        q_table[(state, action)] += alpha * td_error\n        total_reward += reward\n        state = next_state\n    episode_returns.append(total_reward)\n\ndef greedy_action_from_q(state):\n    if env.is_terminal(state):\n        return 'TERM'\n    return max(env.actions, key=lambda a: q_table[(state, a)])\n\nq_value_grid = np.array([[max(q_table[((r, c), a)] for a in env.actions) if not env.is_terminal((r, c)) else 0.0 for c in range(env.cols)] for r in range(env.rows)])\nq_policy_grid = np.array([[greedy_action_from_q((r, c)) for c in range(env.cols)] for r in range(env.rows)])\nnp.round(q_value_grid, 3), q_policy_grid, np.mean(episode_returns[-50:])\n

## Step 4 - Compare exact planning and sampled control\n\nOn this small MDP, value iteration gives the exact optimal policy because the model is known. Q-learning reaches a similar policy by bootstrapping from experience alone.\n

In [ ]:
print('Value-iteration greedy policy:')\nprint(policy_grid)\nprint('')\nprint('Q-learning greedy policy:')\nprint(q_policy_grid)\nprint('')\nprint(f'Average return over the last 50 episodes: {np.mean(episode_returns[-50:]):.3f}')\n

## Summary\n\nThis notebook showed the same control problem from two viewpoints. Value iteration used a known model and Bellman optimality updates. Q-learning learned action values directly from sampled experience with an $\varepsilon$-greedy exploration policy.\n

## Exercises\n\n- Change the trap location and predict how the optimal policy changes before rerunning the notebook.\n- Replace Q-learning with SARSA and compare the learned policy under the same exploration rate.\n- Increase the step penalty magnitude and explain how it changes the value surface.\n